# Module 14 — Iterators, Generators, and Lazy Pipelines

## Exercise 14.2 — A pipeline over a file larger than you want in memory

Generates a large log file, then processes it two ways and measures both.
Run:  python ex02_pipeline.py            (default ~50 MB)
      python ex02_pipeline.py --big      (~500 MB -- check your disk first)

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.

---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 2. Generator functions

Any function containing `yield` is a generator function. Calling it **runs
nothing** — it returns a generator object.

In [ ]:
def countdown(n: int):
    print("starting")            # does NOT run on the call
    while n > 0:
        yield n
        n -= 1
    print("done")

gen = countdown(3)               # nothing printed
next(gen)                         # 'starting', then 3
next(gen)                         # 2

`yield` **suspends** the function: locals, instruction pointer, and the whole
frame are preserved. `next()` resumes exactly where it stopped. That suspended
frame is the mental image to carry — it is also how `await` works (Module 22).

### Generators are the easiest way to write `__iter__`

Compare this with Module 09's iterator class:

In [ ]:
class Countdown:
    def __init__(self, start: int) -> None:
        self.start = start

    def __iter__(self):
        current = self.start      # a LOCAL, so each call gets fresh state
        while current > 0:
            yield current
            current -= 1

Two `for` loops both work, because each call to `__iter__` creates a new
generator with its own locals. That is the fix for the one-shot bug, and it is
free.

### `yield from`

In [ ]:
def flatten(nested):
    for item in nested:
        if isinstance(item, list):
            yield from flatten(item)      # delegate, recursively
        else:
            yield item

`yield from x` is not just `for i in x: yield i` — it also forwards `send`,
`throw` and `close`, and propagates the sub-generator's return value. For plain
iteration the loop is equivalent; for coroutines it is not.

### Generator expressions

In [ ]:
squares = (x * x for x in range(1_000_000))     # lazy, ~200 bytes
squares = [x * x for x in range(1_000_000)]     # eager, ~40 MB

sum(x * x for x in data)                         # parens optional as sole arg
any(line.startswith("ERROR") for line in fh)     # short-circuits

**Use a generator expression when the values are consumed once.** Use a list
when you need to index, re-iterate, or take `len()`.

---

## Concept 3. Pipelines

The technique that makes this module worth its time. Each stage is lazy; the
data flows through one item at a time.

In [ ]:
def read_lines(path):
    with open(path, encoding="utf-8") as fh:
        yield from fh

def parse(lines):
    for line in lines:
        parts = line.rstrip("\n").split("\t")
        if len(parts) == 4:
            yield {"ts": parts[0], "level": parts[1],
                   "user": parts[2], "msg": parts[3]}

def only(records, level):
    for r in records:
        if r["level"] == level:
            yield r

def summarise(records, limit):
    for r in islice(records, limit):
        yield f"{r['ts']} {r['user']}: {r['msg']}"

# nothing has run yet
pipeline = summarise(only(parse(read_lines("50gb.log")), "ERROR"), 10)

for line in pipeline:      # NOW it runs, one line at a time
    print(line)

Memory: one line. Work done: it stops after finding ten errors, even if the file
is 50 GB and the tenth error is on line 900.

**Three properties that fall out:**

1. **Constant memory**, regardless of input size.
2. **Early termination** — `break` at any point stops all upstream work.
3. **Composability** — any stage can be inserted, removed, or reordered without
   touching the others.

### The `with` trap in a generator

In [ ]:
def read_lines(path):
    with open(path) as fh:
        yield from fh          # the file stays open while the generator lives

If the consumer abandons the generator, the `with` block exits when the
generator is garbage collected — which is *usually* immediate under CPython
refcounting and *not guaranteed* (Module 02). For long-lived programs, close it
explicitly or use `contextlib.closing`. This is a real source of "too many open
files" in production.

---

## Concept 4. `itertools`

The composable toolkit. Everything here is lazy.

In [ ]:
from itertools import (
    chain, islice, tee, cycle, repeat, count,
    groupby, takewhile, dropwhile, filterfalse, compress,
    accumulate, pairwise, product, permutations, combinations, zip_longest,
)

chain(a, b, c)                  # concatenate iterables
chain.from_iterable(nested)     # flatten one level -- the O(n) way
islice(it, 10)                  # a slice of an iterator
islice(it, 5, 15)
takewhile(lambda x: x < 100, it)   # stop at the first failure
dropwhile(lambda x: x < 100, it)   # skip until the first success
accumulate(nums)                    # running totals
pairwise("abcd")                    # ('a','b'), ('b','c'), ('c','d')  3.10+
groupby(sorted(rows, key=f), key=f) # group CONSECUTIVE equal keys
zip_longest(a, b, fillvalue=0)
count(1)                            # 1, 2, 3, ... infinite

**`groupby` requires sorted input.** It groups *consecutive* equal keys, like
Unix `uniq`. Unsorted input silently produces many small groups instead of one
per key — a quiet wrong answer, not an error. If you cannot sort (it is an
infinite stream, or sorting is too expensive), use `defaultdict(list)` instead.

**`tee` is not free.** `tee(it, 2)` buffers everything one branch has consumed
and the other has not. If one branch runs ahead, the buffer grows to that gap.
Two independent passes over a list are usually cheaper.

**Flattening:**

In [ ]:
sum(lists, [])                          # O(n^2). Never.
list(chain.from_iterable(lists))        # O(n). Always.

---

## Concept 6. When *not* to be lazy

Laziness is not free, and it is not always right.

| Situation | Use |
|---|---|
| Need `len()` | a list |
| Need to iterate twice | a list |
| Need indexing or slicing | a list |
| Small data (under ~1000 items) | a list — clearer, and faster |
| Result feeds a C library (NumPy, pandas) | a list or array |
| Data larger than memory | a generator |
| Infinite or unbounded stream | a generator |
| Early termination likely | a generator |
| Expensive per-item work, may not need all | a generator |

**The debugging cost is real.** A generator pipeline shows you nothing until it
runs, a traceback points at the *consumption* site rather than the definition,
and you cannot inspect intermediate state in a debugger without consuming it.
`list()` a stage temporarily when debugging.

**The exhaustion bug is the one that bites.** Passing a generator to a function
that iterates it twice produces an empty second pass and no error at all
(Module 05's exercise). If a function must iterate twice, it should take a
`Sequence`, not an `Iterable`, and say so in its signature.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: The iterator protocol
- Section 2: Generator functions
- Section 3: Pipelines
- Section 4: `itertools`
- Section 5: Generators as coroutines
- Section 6: When *not* to be lazy

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

import random
import sys
import tempfile
import tracemalloc
from collections.abc import Iterable, Iterator
from itertools import islice
from pathlib import Path
from typing import Any

LEVELS = ["DEBUG"] * 60 + ["INFO"] * 30 + ["WARN"] * 8 + ["ERROR"] * 2
USERS = [f"u{i}" for i in range(500)]

---

## `generate`

_generate_

In [ ]:
def generate(path: Path, lines: int) -> None:
    rng = random.Random(0)
    with path.open("w", encoding="utf-8") as fh:
        for i in range(lines):
            fh.write(f"2026-08-{i % 28 + 1:02d}T12:00:00\t{rng.choice(LEVELS)}\t"
                     f"{rng.choice(USERS)}\trequest {i} completed in "
                     f"{rng.randrange(1, 5000)}ms\n")

---

## `eager_top_errors`

Reads the WHOLE file into memory, then filters. Do not run this on

In [ ]:
def eager_top_errors(path: Path, limit: int) -> list[str]:
    """Reads the WHOLE file into memory, then filters. Do not run this on
    --big unless you want to meet the OOM killer."""
    lines = path.read_text(encoding="utf-8").splitlines()
    records = [dict(zip(("ts", "level", "user", "msg"), line.split("\t")))
               for line in lines]
    errors = [r for r in records if r["level"] == "ERROR"]
    return [f"{r['ts']} {r['user']}: {r['msg']}" for r in errors[:limit]]

---

## `read_lines`

Yield lines. Note the `with` trap from the README: this generator holds

In [ ]:
def read_lines(path: Path) -> Iterator[str]:
    """Yield lines. Note the `with` trap from the README: this generator holds
    the file open for as long as it lives. Decide how to handle that and write
    down why."""
    raise NotImplementedError

---

## `parse`

Parse tab-separated lines into dicts. Skip malformed lines rather than

In [ ]:
def parse(lines: Iterable[str]) -> Iterator[dict[str, str]]:
    """Parse tab-separated lines into dicts. Skip malformed lines rather than
    raising -- but COUNT them, because silently dropping data is how a
    pipeline lies to you. How will you return that count from a generator?
    (There are three answers. Pick one and say why.)"""
    raise NotImplementedError

---

## `only_level`

_only level_

In [ ]:
def only_level(records: Iterable[dict[str, str]], level: str) -> Iterator[dict[str, str]]:
    raise NotImplementedError

---

## `format_records`

_format records_

In [ ]:
def format_records(records: Iterable[dict[str, str]]) -> Iterator[str]:
    raise NotImplementedError

---

## `lazy_top_errors`

Compose the four stages and take the first `limit` results.

In [ ]:
def lazy_top_errors(path: Path, limit: int) -> list[str]:
    """Compose the four stages and take the first `limit` results."""
    raise NotImplementedError

---

## `measure`

Run both versions under tracemalloc and print peak memory and time.

In [ ]:
def measure(path: Path, limit: int = 10) -> None:
    """Run both versions under tracemalloc and print peak memory and time.

    Predict BEFORE running:
      - roughly what will the eager peak be, relative to the file size?
      - roughly what will the lazy peak be?
      - which is FASTER for limit=10, and why?
      - which is faster for limit=1_000_000 (more errors than exist)?
    That last pair is the interesting one -- laziness is not universally
    faster, and knowing when it is not is the point.
    """
    raise NotImplementedError

---

## `count_by_user`

Count ERROR lines per user across the WHOLE file.

In [ ]:
def count_by_user(path: Path, level: str) -> dict[str, int]:
    """Count ERROR lines per user across the WHOLE file.

    This one CANNOT terminate early -- it must see every line. Does laziness
    still help? Measure it and explain the result. (Hint: what is resident at
    any moment, and how big is the result?)
    """
    raise NotImplementedError

---

## `busiest_hour`

Find the hour with the most log lines.

In [ ]:
def busiest_hour(path: Path) -> tuple[str, int]:
    """Find the hour with the most log lines.

    Now a harder question: this needs a full pass AND a grouping. Write it two
    ways -- with a Counter, and with itertools.groupby -- and say why groupby
    is the wrong tool here even though it looks like a grouping problem.
    """
    raise NotImplementedError

---

## `main`

_main_

In [ ]:
def main() -> int:
    lines = 5_000_000 if "--big" in sys.argv else 500_000
    with tempfile.TemporaryDirectory() as td:
        path = Path(td) / "app.log"
        print(f"generating {lines:,} lines...")
        generate(path, lines)
        size_mb = path.stat().st_size / 1_048_576
        print(f"  {size_mb:.1f} MB\n")

        measure(path)
        print()
        print("errors by user (top 5):",
              sorted(count_by_user(path, "ERROR").items(),
                     key=lambda kv: -kv[1])[:5])
        print("busiest hour:", busiest_hour(path))
    return 0

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    raise SystemExit(main())

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.